## Create QA Pairs

After running `create_figures_batch.py` -- this will create the QA stuffs.

In [11]:
save_dir_in = "~/Dropbox/jcdl_followup/synthetic_figures/"

save_dir_out_json = "~/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/"

In [12]:
import json
from glob import glob
import os
from copy import deepcopy
from importlib import reload
import numpy as np

import utils.figure_level_qa_utils
reload(utils.figure_level_qa_utils)

from utils.plot_qa_utils import init_qa_pairs
from utils.figure_level_qa_utils import figure_level_qa
from utils.general_plot_level_qa_utils import q_errorbars_existance_lines
from utils.misc_data_utils import NumpyEncoder
from functools import partial

stats = {'minimum':np.min, 'maximum':np.max, 'median':np.median, 'mean':np.mean}
linestyles = ['-', '--', ':'] # only use a subset of the linestyles

In [13]:
save_dir_in = os.path.expanduser(save_dir_in)
save_dir_out_json = os.path.expanduser(save_dir_out_json)

if not os.path.exists(save_dir_out_json):
    os.mkdir(save_dir_out_json)
    print('made:', save_dir_out_json)

In [14]:
files_in = glob(save_dir_in + 'jsons/*.json')
files_in[:3]

['/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/jsons/Picture_000121.json',
 '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/jsons/Picture_000064.json',
 '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/jsons/Picture_000176.json']

In [15]:
# get all plot types that have been used
plot_types = np.array([])
for fi in files_in:
    with open(fi, 'r') as f:
        data = json.load(f)
        data = json.loads(data)

    # overall panels
    for k,v in data.items():
        if 'plot' in k:
            plot_types = np.unique(np.concatenate([plot_types,[v['type']]]))
plot_types = plot_types.tolist()

plot_types

['contour', 'histogram', 'line', 'scatter']

In [16]:
import utils.histogram_plot_qa_utils
reload(utils.histogram_plot_qa_utils)

from utils.histogram_plot_qa_utils import q_nbars_hist_plot_plotnums, q_stats_hists, q_relationship_histograms

def plot_level_histogram_qa(data, qa_pairs, iplot, stats, 
                          line_list = ['random','linear','gaussian mixture model'], 
                          verbose_qa=False):
    ######## L1 #######
    # number of bars
    qa_pairs = q_nbars_hist_plot_plotnums(data, qa_pairs, plot_num = iplot, 
                                            use_words=True, verbose=verbose_qa)
    ###### L2 #######
    # stats items
    for k,v in stats.items():
        qa_pairs = q_stats_hists(data, qa_pairs, stat={k:v}, plot_num=iplot, 
                                    use_words=True, verbose=verbose_qa)
        
    line_list = ['random','linear','gaussian mixture model']
    qa_pairs = q_relationship_histograms(data, qa_pairs, plot_num = iplot, 
            return_qa=True, use_words=True, 
            use_list=True, 
            line_list = line_list,
        verbose=verbose_qa)
    return qa_pairs

In [17]:
import utils.scatter_plot_qa_utils
reload(utils.scatter_plot_qa_utils)
from utils.scatter_plot_qa_utils import q_npoints_scatter_plot_plotnums, \
    q_stats_scatters, q_relationship_scatters

def plot_level_scatter_qa(data, qa_pairs, iplot, stats, 
                          line_list = ['random','linear','gaussian mixture model'], 
                          verbose_qa=False):
    ######### L1 #########
    qa_pairs = q_npoints_scatter_plot_plotnums(data, qa_pairs, 
                                        plot_num = iplot, 
                                        verbose=verbose_qa)
    ######### L2 #########
    # stats items
    for k,v in stats.items(): # for all stats
        for axis in ['x', 'y', 'color']:
            qa_pairs = q_stats_scatters(data, qa_pairs, stat={k:v}, 
                                        plot_num=iplot, use_words=True, 
                                        verbose=verbose_qa, axis=axis)
    # ###### L3 ######
    # type of distribution
    for axis in ['color', 'x/y']:
        qa_pairs = q_relationship_scatters(data, qa_pairs, plot_num = iplot, axis=axis,
                    return_qa=True, use_words=True, 
                    use_list=True, 
                line_list = line_list,
                verbose=verbose_qa)
        
    return qa_pairs

In [18]:
import utils.linear_plot_qa_utils
reload(utils.linear_plot_qa_utils)

from utils.linear_plot_qa_utils import q_nlines_plot_plotnums, q_stats_lines, \
    q_relationship_lines, q_linestyles_lines

def plot_level_line_qa(data, qa_pairs, iplot, stats, verbose_qa=False):
    ######### L1 #########
    qa_pairs = q_nlines_plot_plotnums(data, qa_pairs, 
                                        plot_num = iplot, 
                                        verbose=verbose_qa)
    # note: line colors don't work right now
    # qa_pairs = q_colors_lines(data, qa_pairs, 
    #                                   plot_num = iplot, 
    #                                   verbose=verbose_qa)    
    qa_pairs = q_linestyles_lines(data, qa_pairs, plot_num=iplot, 
                                    use_words=True, use_list = True,
                        linestyle_list=linestyles, verbose=verbose_qa)      
    ######### L2 #########
    # stats items
    for k,v in stats.items(): # for all stats
        qa_pairs = q_stats_lines(data, qa_pairs, stat={k:v}, 
                                    plot_num=iplot, use_words=True, 
                                    verbose=verbose_qa)
    ######## L3 #########
    for use_list in [True,False]:
        for use_nlines in [True, False]:
            qa_pairs = q_relationship_lines(data, qa_pairs, plot_num=iplot, use_words=True, 
                                        use_nlines = use_nlines, 
                                        verbose=verbose_qa, use_list=use_list)
            
    return qa_pairs

In [19]:
import utils.contour_plot_qa_utils
reload(utils.contour_plot_qa_utils)

from utils.contour_plot_qa_utils import q_stats_contours, q_relationship_contour, \
    q_contour_plot_image_or_lines

def plot_level_contour_qa(data, qa_pairs, iplot, stats,                           
        line_list = ['random','linear','gaussian mixture model'], 
        verbose_qa=False):
    ######### L1 #########
    qa_pairs = q_contour_plot_image_or_lines(data, qa_pairs, 
                                        plot_num = iplot, 
                                        verbose=verbose_qa)
  
    ######### L2 #########
    # stats items
    for k,v in stats.items(): # for all stats
        for axis in ['x', 'y', 'color']:
            qa_pairs = q_stats_contours(data, qa_pairs, stat={k:v}, 
                                        plot_num=iplot, use_words=True, 
                                        verbose=verbose_qa, axis=axis)
    # ###### L3 ######
    # type of distribution
    for axis in ['color', 'x/y']:
        qa_pairs = q_relationship_contour(data, qa_pairs, plot_num = iplot, axis=axis,
                    return_qa=True, use_words=True, 
                    use_list=True, 
                line_list = line_list,
                verbose=verbose_qa)
            
    return qa_pairs

In [24]:
for fi in files_in:
    with open(fi, 'r') as f:
        data = json.load(f)
        data = json.loads(data)

    # overall panels
    qa_pairs = init_qa_pairs()
    #plot_types = []; 
    plot_nums = []
    for k,v in data.items():
        if 'plot' in k:
            #plot_types.append(v['type'])
            plot_nums.append(int(k.split('plot')[-1]))
            # print('here')
            # import sys; sys.exit()

    # figure level
    qa_pairs = figure_level_qa(data, deepcopy(qa_pairs), plot_types, verbose=False)
    #import sys; sys.exit()
    # individual plots -- updated?
    for iplot in plot_nums:
        # general plot-level questions
        ##### L2 ######
        for axis in ['x', 'y']:
            qa_pairs = q_errorbars_existance_lines(data, qa_pairs, 
                                              plot_num = iplot, 
                                              verbose=False, 
                                              axis=axis)

        if data['plot'+str(iplot)]['type'] == 'line': # line plots
            #print('----------- LINE -----------')
            qa_pairs = plot_level_line_qa(data, qa_pairs, iplot, stats, 
                                          verbose_qa=False)
        elif data['plot'+str(iplot)]['type'] == 'scatter':
            #print('----------- SCATTER -----------')
            qa_pairs = plot_level_scatter_qa(data, qa_pairs, iplot, stats, 
                          line_list = ['random','linear','gaussian mixture model'], 
                          verbose_qa=False)
        elif data['plot'+str(iplot)]['type'] == 'histogram':
            #print('----------- HISTOGRAM -----------')
            qa_pairs = plot_level_histogram_qa(data, qa_pairs, iplot, stats, 
                          line_list = ['random','linear','gaussian mixture model'], 
                          verbose_qa=False)
        elif data['plot'+str(iplot)]['type'] == 'contour':
            qa_pairs = plot_level_contour_qa(data, qa_pairs, iplot, stats, 
                          verbose_qa=False)
            #import sys; sys.exit()
    #import sys; sys.exit()
    # update
    json_out = deepcopy(data)
    json_out['VQA'] = deepcopy(qa_pairs)
    # dump full data
    #dumped = json.dumps(json_out, cls=NumpyEncoder(verbose=True))
    dumped = json.dumps(json_out, cls=partial(NumpyEncoder, verbose=True))
    fiout = fi.split('/')[-1].removesuffix('.json') + '_qa.json'
    #print(save_dir_out_json + fiout)
    # with open(save_dir_out_json + fiout, 'w') as f:
    #     json.dump(dumped, f)

print('!!!!! DONE !!!!')

!!!!! DONE !!!!


In [21]:
fi

'/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/jsons/Picture_000159.json'

In [22]:
data[k]['data from plot']

{'data': {'image': 'non serializable entry'},
 'color bar': 'non serializable entry',
 'color bar params': {'side': 'right',
  'pad': 0.0232758731691695,
  'size': '11%',
  'axis side': 'right',
  'label prob': 0.5}}